# California Housing Price Prediction

Predict median house values using the California Housing dataset (20,640 samples, 8 numeric features).

This demonstrates:
- Loading CSV data with `affon:data`
- Feature standardization and data pipeline
- Training a dense neural network with SGD optimizer

In [7]:
import { relu, sgd, clear_grad, grad } from 'affon:compute/legacy'
import { read } from 'affon:dataset'
import nn from 'affon:nn/legacy'


## Load and Prepare Data

The dataset has 20,640 samples with 8 numeric features and 1 categorical (`ocean_proximity`).
We drop the categorical column, handle missing values, and standardize all columns.

In [8]:
const dataset = read('california_housing.csv')
  .drop('ocean_proximity')
  .dropna()
  .standardize('longitude', 'latitude', 'housing_median_age', 'total_rooms',
               'total_bedrooms', 'population', 'households', 'median_income',
               'median_house_value')
  .sample(500)
  .shuffle()

console.log('Columns:', dataset.columns)

Columns: [ longitude, latitude, housing_median_age, total_rooms, total_bedrooms, population, households, median_income, median_house_value, ocean_proximity ]


In [9]:
const features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                  'total_bedrooms', 'population', 'households', 'median_income']

const { X, y } = dataset
  .features(...features)
  .target('median_house_value')
  .toTensors({ dtype: 'f32' })

console.log('X:', X.shape, X.dtype, 'y:', y.shape, y.dtype)


X: [ 500, 8 ] f32 y: [ 500 ] f32


## Define Model

A 3-layer dense network: 8 -> 32 -> 16 -> 1 with ReLU activations.

In [10]:
const model = nn.Sequential(
  nn.Linear(8, 32),
  relu,
  nn.Linear(32, 16),
  relu,
  nn.Linear(16, 1)
)

const criterion = nn.MSELoss()
const params = model.parameters
const step = sgd({ lr: 0.01 })

console.log('Parameters:', params.length)


Parameters: 6


## Train

In [11]:
const epochs = 500

for (let epoch = 0; epoch < epochs; epoch++) {
  clear_grad(params)
  const pred = model(X)
  const loss = criterion(pred, y)
  grad(loss, params)
  step(params)

  if (epoch % 100 === 0 || epoch === epochs - 1) {
    console.log(`Epoch ${epoch}: loss = ${loss.item().toFixed(4)}`)
  }
}


Epoch 0: loss = 2.0553
Epoch 100: loss = 0.2709
Epoch 200: loss = 0.2555
Epoch 300: loss = 0.2482
Epoch 400: loss = 0.2432
Epoch 499: loss = 0.2392


## Results

In [12]:
// Verify model works for inference after training
const preds = model(X)
console.log('Prediction shape:', preds.shape)
console.log('Final MSE:', criterion(preds, y).item().toFixed(4))

Prediction shape: [ 500, 1 ]
Final MSE: 0.2391
